# Taxonomy Extraction

Run the enriched taxonomy prompt on papers predicted as POSITIVE by the classifier.
Uses the `taxonomy` task config from `llm_labeller.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

In [3]:
from src.taxonomy.providers import (
    claude_batch_submit,
    claude_batch_results,
    mistral_batch_submit,
    mistral_batch_results,
    google_batch_submit,
    google_batch_results,
    openai_batch_submit,
    openai_batch_results,
    label_papers_async,
)

c:\Users\samma\Documents\Programmation\UdeM\nlp-benchmark-taxonomy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load positive predictions

In [4]:
predictions = pd.read_parquet(
    DATA_DIR / "classifier" / "predictions" / "inference.parquet"
)

# Keep only predicted positives
positives = predictions[predictions["predicted"] == 1].copy()
print(f"Positive predictions: {len(positives):,} papers")


Positive predictions: 3,738 papers


In [5]:
anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")

# Filter to positives in the anthology
positives_paper = anthology[anthology["bibkey"].isin(positives["bibkey"])]
print(f"Positive predictions in anthology: {len(positives_paper):,} papers")

Positive predictions in anthology: 3,738 papers


## 2. Submit batch classification

### Gemini

In [ ]:
job_id_google = google_batch_submit(positives_paper, task="taxonomy")
job_id_google

In [ ]:
results_google = google_batch_results(job_id_google, positives_paper, task="taxonomy")
results_google.to_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "google.parquet", index=False
)

### Claude

In [ ]:
job_id = claude_batch_submit(positives_paper, task="taxonomy")
job_id

In [ ]:
results = claude_batch_results(job_id, positives_paper, task="taxonomy")
results.to_parquet(DATA_DIR / "taxonomy" / "per_llm" / "claude.parquet", index=False)

### Deepseek

In [ ]:
# DeepSeek — async concurrent (no batch API available)
results_deepseek = await label_papers_async(
    positives_paper,
    provider="deepseek",
    task="taxonomy",
    max_concurrent=20,
)

In [ ]:
results_deepseek.to_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "deepseek.parquet", index=False
)

### Mistral

In [ ]:
mistral_job_id = mistral_batch_submit(positives_paper, task="taxonomy")
mistral_job_id

In [ ]:
results_mistral = mistral_batch_results(
    mistral_job_id, positives_paper, task="taxonomy"
)
results_mistral.to_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral.parquet", index=False
)

## 3. Analyse Predictions

In [ ]:
results_deepseek = pd.read_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "deepseek.parquet"
)
results_claude = pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / "claude.parquet")
results_mistral = pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / "mistral.parquet")
results_google = pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / "google.parquet")

In [ ]:
from collections import Counter

LABEL_COLS = {
    "Google": "google_label",
    "DeepSeek": "deepseek_label",
    "Mistral": "mistral_label",
}

results_merged = (
    results_google[
        [
            "bibkey",
            "google_label",
            "google_benchmark_names",
            "google_task_type",
            "google_input_type",
            "google_label_type",
            "google_domain",
            "google_languages",
            "google_is_multitask",
        ]
    ]
    .merge(
        results_deepseek[
            [
                "bibkey",
                "deepseek_label",
                "deepseek_benchmark_names",
                "deepseek_task_type",
                "deepseek_input_type",
                "deepseek_label_type",
                "deepseek_domain",
                "deepseek_languages",
                "deepseek_is_multitask",
            ]
        ],
        on="bibkey",
    )
    .merge(
        results_mistral[
            [
                "bibkey",
                "mistral_label",
                "mistral_benchmark_names",
                "mistral_task_type",
                "mistral_input_type",
                "mistral_label_type",
                "mistral_domain",
                "mistral_languages",
                "mistral_is_multitask",
            ]
        ],
        on="bibkey",
    )
    .dropna(subset=list(LABEL_COLS.values()))
)


def _majority_and_dissenter(row):
    votes = {name: row[col] for name, col in LABEL_COLS.items()}
    counts = Counter(votes.values())
    winner, n = counts.most_common(1)[0]
    dissenters = [k for k, v in votes.items() if v != winner]
    return pd.Series(
        {
            "majority_label": winner,
            "dissenter": dissenters[0] if dissenters else None,
        }
    )


results_merged = pd.concat(
    [results_merged, results_merged.apply(_majority_and_dissenter, axis=1)],
    axis=1,
)

In [ ]:
task_map = pd.read_csv(DATA_DIR / "mappings" / "task_type_mapping.csv")
mapping = dict(zip(task_map["raw"], task_map["canonical"]))

for prefix in ["google", "deepseek", "mistral"]:
    col = f"{prefix}_task_type"
    results_merged[col] = results_merged[col].map(mapping).fillna(results_merged[col])


In [ ]:
results_merged.to_parquet(DATA_DIR / "taxonomy" / "merged.parquet", index=False)

### Cohens Kappa

In [ ]:
kappa = cohen_kappa_score(
    results_merged["google_label"],
    results_merged["deepseek_label"],
)
kappa_mistral = cohen_kappa_score(
    results_merged["deepseek_label"],
    results_merged["mistral_label"],
)

kappa_claude_mistral = cohen_kappa_score(
    results_merged["google_label"],
    results_merged["mistral_label"],
)

print(f"Cohen's kappa (Deepseek vs Google): {kappa:.3f}")
print(f"Cohen's kappa (Deepseek vs Mistral): {kappa_mistral:.3f}")
print(f"Cohen's kappa (Google vs Mistral): {kappa_claude_mistral:.3f}")

## 4. Submit batch taxonomy extraction

In [6]:
positives_llm = pd.read_parquet(DATA_DIR / "taxonomy" / "merged.parquet")
anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")
anthology = anthology[anthology["year"] < 2026]

common = positives_llm.columns.intersection(anthology.columns).difference(["bibkey"])
positives_llm = positives_llm.merge(anthology.drop(columns=common), on="bibkey")
positives_llm = positives_llm[positives_llm["year"] < 2026]
positives_llm = positives_llm[positives_llm["majority_label"] == "POSITIVE"]

In [7]:
already_labeled = pd.read_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_extraction.parquet"
)["bibkey"].unique()
# positives_llm = positives_llm.sample(300, random_state=42)
positives_llm = positives_llm[~positives_llm["bibkey"].isin(already_labeled)]

### Mistral

In [ ]:
mistral_job_id = mistral_batch_submit(positives_llm, task="benchmark_record_extraction")
mistral_job_id

In [8]:
mistral_job_id = "c8ab33fa-f34f-4bc2-a323-611b7062eddb"

In [9]:
import json

results_mistral = mistral_batch_results(
    mistral_job_id, positives_llm, task="benchmark_record_extraction"
)

# Flatten records: one row per record
if "mistral_records" in results_mistral.columns:
    records_list = []
    for idx, row in results_mistral.iterrows():
        if isinstance(row["mistral_records"], list):
            for rec in row["mistral_records"]:
                rec_dict = {
                    "bibkey": row["bibkey"],
                    "subtasks_not_inferable": row.get(
                        "mistral_subtasks_not_inferable", False
                    ),
                    **rec,
                }
                records_list.append(rec_dict)
    results_mistral = pd.DataFrame(records_list)

    # Convert all list/dict columns to JSON strings for parquet compatibility
    for col in results_mistral.columns:
        if results_mistral[col].dtype == "object":
            # Check if column contains lists or mixed types
            try:
                if any(
                    isinstance(x, (list, dict))
                    for x in results_mistral[col]
                    if x is not None
                ):
                    results_mistral[col] = results_mistral[col].apply(
                        lambda x: (
                            json.dumps(x)
                            if isinstance(x, (list, dict))
                            else str(x)
                            if x is not None
                            else None
                        )
                    )
            except Exception:
                pass

    # Ensure numeric/mixed columns are strings to avoid Arrow type inference issues
    for col in ["label_space_size", "languages", "evidence_passages"]:
        if col in results_mistral.columns:
            results_mistral[col] = results_mistral[col].astype(str)


Validation error for rostami-etal-2025-politisky24: 1 validation error for ExtractionResult
records
  Field required [type=missing, input_value={'reasoning': "1. Emittin... units (e.g., whether "}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
Validation error for kurihara-etal-2022-jglue: 3 validation errors for ExtractionResult
records.0.input_cardinality
  Input should be 'single_text', 'pair_of_texts', 'text_with_target', 'sequence_of_turns' or 'text_with_context' [type=enum, input_value='unknown', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum
records.0.input_unit
  Input should be 'token', 'span', 'sentence', 'short_text', 'paragraph', 'document', 'dialog_turn' or 'dialog' [type=enum, input_value='unknown', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum
records.0.output_cardinality
  Input should be 'single_label', 'multi_label', 'ordinal', 'hierar

In [10]:
path = DATA_DIR / "taxonomy" / "per_llm" / "mistral_extraction_full.parquet"

# 1. Vérifier si le fichier existe
if path.exists():
    existing_df = pd.read_parquet(path)
    updated_df = pd.concat([existing_df, results_mistral], ignore_index=True)
else:
    updated_df = results_mistral
updated_df.to_parquet(path, index=False)

### DeepSeek

In [ ]:
# DeepSeek — async concurrent (no batch API available)
results_deepseek = await label_papers_async(
    positives_llm,
    provider="deepseek",
    task="benchmark_record_extraction",
    max_concurrent=20,
)

In [ ]:
if "deepseek_records" in results_deepseek.columns:
    records_list = []
    for _, row in results_deepseek.iterrows():
        if isinstance(row["deepseek_records"], list):
            for rec in row["deepseek_records"]:
                records_list.append(
                    {
                        "bibkey": row["bibkey"],
                        "subtasks_not_inferable": row.get(
                            "deepseek_subtasks_not_inferable", False
                        ),
                        **rec,
                    }
                )
    results_deepseek = pd.DataFrame(records_list)

# Sérialise listes/dicts, puis force toutes les colonnes object en str
for col in results_deepseek.columns:
    if results_deepseek[col].dtype == "object":
        results_deepseek[col] = results_deepseek[col].apply(
            lambda x: (
                json.dumps(x)
                if isinstance(x, (list, dict))
                else (str(x) if x is not None else None)
            )
        )

results_deepseek.to_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "deepseek_extraction.parquet", index=False
)


## 5. Batch Coarse labels

In [ ]:
anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")
already_labeled = pd.read_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_extraction.parquet"
)["bibkey"].unique()

to_evaluate = anthology[anthology["bibkey"].isin(already_labeled)]


### OpenAI

In [ ]:
openai_job_id = openai_batch_submit(to_evaluate, task="coarse_task_classification")
openai_job_id

In [ ]:
openai_job_id = "batch_69ea737b8ef481908195c560166baf55"

In [ ]:
results_openai = openai_batch_results(
    openai_job_id, to_evaluate, task="coarse_task_classification"
)
results_openai.to_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "openai_coarse.parquet", index=False
)

## 5. Clustering

In [ ]:
import numpy as np
from sklearn.metrics import adjusted_rand_score

from src.taxonomy.embedding import compute_embeddings


In [ ]:
mistral_taxonomy = pd.read_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_extraction.parquet"
)
deepseek_taxonomy = pd.read_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "deepseek_extraction.parquet"
)
print(
    f"Après chargement     — mistral: {len(mistral_taxonomy):,}  deepseek: {len(deepseek_taxonomy):,}"
)

task_types = pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / "google.parquet")[
    ["bibkey", "google_task_type", "title", "abstract"]
]
mistral_taxonomy = mistral_taxonomy.merge(task_types, on="bibkey")
deepseek_taxonomy = deepseek_taxonomy.merge(task_types, on="bibkey")
print(
    f"Après merge task_types — mistral: {len(mistral_taxonomy):,}  deepseek: {len(deepseek_taxonomy):,}"
)

openai_coarse = pd.read_parquet(
    DATA_DIR / "taxonomy" / "per_llm" / "openai_coarse.parquet"
)
mistral_taxonomy = mistral_taxonomy.merge(
    openai_coarse[["bibkey", "openai_coarse_task"]],
    on="bibkey",
)
deepseek_taxonomy = deepseek_taxonomy.merge(
    openai_coarse[["bibkey", "openai_coarse_task"]],
    on="bibkey",
)
print(
    f"Après merge coarse   — mistral: {len(mistral_taxonomy):,}  deepseek: {len(deepseek_taxonomy):,}"
)

# Intersection : bibkeys présents dans les deux
common_bibkeys = set(mistral_taxonomy["bibkey"]) & set(deepseek_taxonomy["bibkey"])
mistral_taxonomy = mistral_taxonomy[mistral_taxonomy["bibkey"].isin(common_bibkeys)]
deepseek_taxonomy = deepseek_taxonomy[deepseek_taxonomy["bibkey"].isin(common_bibkeys)]
print(
    f"Après intersection   — mistral: {len(mistral_taxonomy):,}  deepseek: {len(deepseek_taxonomy):,}  ({len(common_bibkeys):,} bibkeys communs)"
)

# True si le papier a produit plusieurs records (benchmark multi-tâche)
for df in [mistral_taxonomy, deepseek_taxonomy]:
    df["is_multi_record"] = df.groupby("bibkey")["bibkey"].transform("count") > 1

print(
    f"Multi-records        — mistral: {mistral_taxonomy['is_multi_record'].sum():,}  deepseek: {deepseek_taxonomy['is_multi_record'].sum():,}"
)

# Garder uniquement les bibkeys sans multi-records dans les deux
single_bibkeys = set(
    mistral_taxonomy.loc[~mistral_taxonomy["is_multi_record"], "bibkey"]
) & set(deepseek_taxonomy.loc[~deepseek_taxonomy["is_multi_record"], "bibkey"])
mistral_taxonomy = mistral_taxonomy[mistral_taxonomy["bibkey"].isin(single_bibkeys)]
deepseek_taxonomy = deepseek_taxonomy[deepseek_taxonomy["bibkey"].isin(single_bibkeys)]
print(
    f"Après filtre multi   — mistral: {len(mistral_taxonomy):,}  deepseek: {len(deepseek_taxonomy):,}  ({len(single_bibkeys):,} bibkeys)"
)

# Aligner l'ordre des bibkeys
mistral_taxonomy = mistral_taxonomy.sort_values("bibkey").reset_index(drop=True)
deepseek_taxonomy = deepseek_taxonomy.sort_values("bibkey").reset_index(drop=True)

assert (
    mistral_taxonomy["bibkey"].values == deepseek_taxonomy["bibkey"].values
).all(), "Les bibkeys ne sont pas dans le même ordre !"
print("Ordre bibkeys aligné ✓")


### Embedding

In [ ]:
embeddings_mistral = compute_embeddings(
    mistral_taxonomy["paraphrase_task"].astype(str).tolist()
)

embeddings_deepseek = compute_embeddings(
    deepseek_taxonomy["paraphrase_task"].astype(str).tolist()
)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

emb_m = np.array(embeddings_mistral)

# Distribution des distances au centroïde
center = emb_m.mean(axis=0)
dist_to_center = 1 - cosine_similarity(emb_m, center.reshape(1, -1)).ravel()
print(
    f"Dist au centre — mean: {dist_to_center.mean():.3f}, std: {dist_to_center.std():.3f}"
)
print(f"% sous 0.3: {(dist_to_center < 0.3).mean():.1%}")


### HAC

In [ ]:
import plotly.io as pio
from src.taxonomy.hac_utils import (
    EXCLUDED_LABELS,
    build_linkage,
    clustering_metrics,
    cophenetic_table,
    dendrogram_purity,
    make_color_map,
    plot_dendrogram,
)

pio.renderers.default = "browser"

# ── Linkage matrices ──────────────────────────────────────────────────────────
embeddings_arr_m = np.array(embeddings_mistral)
embeddings_arr_d = np.array(embeddings_deepseek)

Z_m = build_linkage(embeddings_arr_m)
Z_d = build_linkage(embeddings_arr_d)

# Palette partagée (mêmes couleurs pour les deux dendrogrammes)
coarse_values_m = mistral_taxonomy["openai_coarse_task"].fillna("Unknown")
coarse_values_d = deepseek_taxonomy["openai_coarse_task"].fillna("Unknown")
all_coarse = pd.concat([coarse_values_m, coarse_values_d])
color_map = make_color_map(all_coarse)


In [ ]:
# ── Dendrogramme Mistral ──────────────────────────────────────────────────────
fig_m = plot_dendrogram(
    embeddings_arr_m,
    mistral_taxonomy,
    coarse_col="openai_coarse_task",
    label_col="openai_coarse_task",
    title="HAC — Mistral (coloré par coarse label)",
    color_map=color_map,
)
fig_m.show(config={"responsive": True, "scrollZoom": True, "displayModeBar": True})
fig_m.write_html("dendrogramme_mistral.html")

In [ ]:
# ── Dendrogramme DeepSeek ─────────────────────────────────────────────────────
fig_d = plot_dendrogram(
    embeddings_arr_d,
    deepseek_taxonomy,
    coarse_col="openai_coarse_task",
    label_col="openai_coarse_task",
    title="HAC — DeepSeek (coloré par coarse label)",
    color_map=color_map,
)
fig_d.show(config={"responsive": True, "scrollZoom": True, "displayModeBar": True})


In [ ]:
N_CLUSTERS = 20

metrics_m = clustering_metrics(
    embeddings_arr_m, Z_m, coarse_values_m, n_clusters=N_CLUSTERS
)
metrics_d = clustering_metrics(
    embeddings_arr_d, Z_d, coarse_values_d, n_clusters=N_CLUSTERS
)

rows = {
    "n_clusters": [metrics_m["n_clusters"], metrics_d["n_clusters"]],
    "ARI vs coarse": [f"{metrics_m['ari']:.3f}", f"{metrics_d['ari']:.3f}"],
    "NMI vs coarse": [f"{metrics_m['nmi']:.3f}", f"{metrics_d['nmi']:.3f}"],
    "Silhouette": [f"{metrics_m['silhouette']:.3f}", f"{metrics_d['silhouette']:.3f}"],
    "Cosine intra": [
        f"{metrics_m['cosine_intra']:.3f}",
        f"{metrics_d['cosine_intra']:.3f}",
    ],
    "Cosine inter": [
        f"{metrics_m['cosine_inter']:.3f}",
        f"{metrics_d['cosine_inter']:.3f}",
    ],
    "Ratio i/e": [
        f"{metrics_m['cosine_ratio']:.3f}",
        f"{metrics_d['cosine_ratio']:.3f}",
    ],
}
summary = pd.DataFrame(rows, index=["Mistral", "DeepSeek"]).T
print(f"k={N_CLUSTERS} clusters (maxclust)\n")
print(summary.to_string())

# ARI entre les deux clusterings
ari_cross = adjusted_rand_score(metrics_m["hac_labels"], metrics_d["hac_labels"])
print(f"\nARI Mistral ↔ DeepSeek (k={N_CLUSTERS}) : {ari_cross:.3f}")


In [ ]:
# On compare les deux arbres contre les coarse labels

dp_m = dendrogram_purity(Z_m, coarse_values_m.to_numpy())
dp_d = dendrogram_purity(Z_d, coarse_values_d.to_numpy())

n_classes = len([c for c in coarse_values_m.unique() if c not in EXCLUDED_LABELS])
baseline = 1 / n_classes

print(f"{'':30} {'Mistral':>10}  {'DeepSeek':>10}  {'baseline':>10}")
print("─" * 64)
print(f"{'Dendrogram Purity':<30} {dp_m:>10.3f}  {dp_d:>10.3f}  {baseline:>10.3f}")

# ── Cophenetic intra / inter ──────────────────────────────────────────────────
ct_m = cophenetic_table(Z_m, coarse_values_m.to_numpy())
ct_d = cophenetic_table(Z_d, coarse_values_d.to_numpy())

merged = ct_m.merge(ct_d, on="family", suffixes=("_mistral", "_deepseek"))
merged = merged.sort_values("ratio_mistral")

print("\nCophenetic intra / inter par famille :")
print(
    f"{'Famille':<30} {'intra_M':>8} {'inter_M':>8} {'ratio_M':>8}  {'intra_D':>8} {'inter_D':>8} {'ratio_D':>8}  {'n':>4}"
)
print("─" * 90)
for _, row in merged.iterrows():
    print(
        f"{row['family']:<30}"
        f" {row['intra_mistral']:>8.3f} {row['inter_mistral']:>8.3f} {row['ratio_mistral']:>8.3f}"
        f"  {row['intra_deepseek']:>8.3f} {row['inter_deepseek']:>8.3f} {row['ratio_deepseek']:>8.3f}"
        f"  {row['n_mistral']:>4.0f}"
    )
print("─" * 90)
g_m = ct_m["intra"].mean() / ct_m["inter"].mean()
g_d = ct_d["intra"].mean() / ct_d["inter"].mean()
print(f"{'GLOBAL':<30} {'':>8} {'':>8} {g_m:>8.3f}  {'':>8} {'':>8} {g_d:>8.3f}")
print("\nratio < 1 → famille bien regroupée  |  ratio > 1 → famille dispersée")


In [ ]:
from scipy.cluster.hierarchy import cophenet, fcluster
from scipy.stats import pearsonr, spearmanr

# ── 1. Corrélation cophénétique croisée ───────────────────────────────────────
coph_m = cophenet(Z_m)
coph_d = cophenet(Z_d)

r_pearson, p_pearson = pearsonr(coph_m, coph_d)
r_spearman, p_spearman = spearmanr(coph_m, coph_d)

print("=== Corrélation cophénétique croisée (Mistral ↔ DeepSeek) ===")
print(f"Pearson  r = {r_pearson:.3f}   (p={p_pearson:.2e})")
print(f"Spearman r = {r_spearman:.3f}   (p={p_spearman:.2e})")
print("(1 = arbres identiques, 0 = aucune corrélation)")

# ── 2. Dendrogram Purity croisé ───────────────────────────────────────────────
# Labels issus de l'arbre de l'autre LLM → mesure si les deux arbres
# regroupent les mêmes papiers ensemble.
labels_m_k = fcluster(Z_m, t=N_CLUSTERS, criterion="maxclust").astype(str)
labels_d_k = fcluster(Z_d, t=N_CLUSTERS, criterion="maxclust").astype(str)

dp_m_on_d = dendrogram_purity(Z_d, labels_m_k, excluded=frozenset())
dp_d_on_m = dendrogram_purity(Z_m, labels_d_k, excluded=frozenset())

# Baseline par permutation : DP croisé attendu sous labels aléatoires
rng = np.random.default_rng(42)
N_SHUFFLE = 50
baseline_m_on_d = np.mean(
    [
        dendrogram_purity(Z_d, rng.permutation(labels_m_k), excluded=frozenset())
        for _ in range(N_SHUFFLE)
    ]
)
baseline_d_on_m = np.mean(
    [
        dendrogram_purity(Z_m, rng.permutation(labels_d_k), excluded=frozenset())
        for _ in range(N_SHUFFLE)
    ]
)

print(
    f"\n=== Dendrogram Purity croisé (k={N_CLUSTERS} clusters, {N_SHUFFLE} shuffles) ==="
)
print(f"{'':40} {'M→D':>8}  {'D→M':>8}")
print("─" * 58)
print(f"{"DP croisé (labels de l'autre LLM)":<40} {dp_m_on_d:>8.3f}  {dp_d_on_m:>8.3f}")
print(
    f"{'Baseline shuffle (aléatoire)':<40} {baseline_m_on_d:>8.3f}  {baseline_d_on_m:>8.3f}"
)
print(
    f"{'Gain vs baseline':<40} {dp_m_on_d - baseline_m_on_d:>+8.3f}  {dp_d_on_m - baseline_d_on_m:>+8.3f}"
)
print()
print("gain > 0 → les deux arbres capturent des structures similaires")
print("gain ≈ 0 → les regroupements sont indépendants")


In [ ]:
print(
    f"Mistral k={N_CLUSTERS}  cluster sizes: {sorted(Counter(labels_m_k).values(), reverse=True)}"
)
print(
    f"DeepSeek k={N_CLUSTERS} cluster sizes: {sorted(Counter(labels_d_k).values(), reverse=True)}"
)

### Clustering

In [ ]:
from src.taxonomy.hac_utils import (
    cut_tree_adaptive,
    plot_taxo_tree,
    plot_dendrogram_adaptive,
)

# -- Adaptive cut on Mistral embeddings --
taxo_tree_m = cut_tree_adaptive(
    Z_m,
    embeddings_arr_m,
    max_size=50,
    min_cluster_size=5,
)

leaf_nodes = taxo_tree_m._leaf_nodes()
print(f"Clusters: {len(leaf_nodes)}")
print(f"Sizes: {sorted([len(n.leaves) for n in leaf_nodes], reverse=True)}")

# -- Visualisation --
fig_tree_m = plot_taxo_tree(
    taxo_tree_m,
    mistral_taxonomy.reset_index(drop=True),
    label_col="openai_coarse_task",
    title="Adaptive HAC Taxonomy -- Mistral",
)
fig_tree_m.show()

In [ ]:
# Dendrogram coloured by adaptive clusters
fig_dendro_adaptive = plot_dendrogram_adaptive(
    embeddings_arr_m,
    mistral_taxonomy.reset_index(drop=True),
    taxo_tree_m,
    coarse_col="openai_coarse_task",
    label_col="paraphrase_task",
    title="Dendrogram — Adaptive HAC Clusters (Mistral)",
)
fig_dendro_adaptive.show()

In [ ]:
# Attach cluster assignments to mistral_taxonomy and save
flat = taxo_tree_m.flat_labels(n=len(mistral_taxonomy))

leaf_nodes = taxo_tree_m._leaf_nodes()
cluster_label_map = {
    cid: df_row["openai_coarse_task"].value_counts().index[0]
    for cid, node in enumerate(leaf_nodes)
    for df_row in [mistral_taxonomy.reset_index(drop=True).iloc[node.leaves]]
}

mistral_taxonomy = mistral_taxonomy.reset_index(drop=True).copy()
mistral_taxonomy["hac_cluster_id"] = flat
mistral_taxonomy["hac_cluster_label"] = mistral_taxonomy["hac_cluster_id"].map(
    cluster_label_map
)

print(mistral_taxonomy[["hac_cluster_id", "hac_cluster_label"]].value_counts().head(20))

In [ ]:
clustering_df = mistral_taxonomy[
    [
        "bibkey",
        "paraphrase_task",
        "google_task_type",
        "judgment_type",
        "hac_cluster_id",
        "hac_cluster_label",
        "title",
        "abstract",
    ]
]

In [ ]:
from scipy.cluster.hierarchy import linkage


def collect_leaves(node):
    if not node.children:
        return [node]
    out = []
    for c in node.children:
        out.extend(collect_leaves(c))
    return out


idx = clustering_df.index[clustering_df["hac_cluster_id"] == 14].tolist()
sub_Z = linkage(embeddings_arr_m[idx], method="average", metric="cosine")
root = cut_tree_adaptive(
    sub_Z, embeddings_arr_m[idx], max_size=6, tau=0.85, min_cluster_size=2
)

for leaf in collect_leaves(root):
    print(f"\n--- n={len(leaf.leaves)}, coh={leaf.coherence:.2f} ---")
    for j in leaf.leaves:
        print(f"  • {clustering_df.iloc[idx[j]]['paraphrase_task'][:140]}")